# Diet Coke Bangalore Distribution Failure — DMAIC Walkthrough

A Lean Six Sigma Green Belt analysis of the real April-2026 Diet Coke aluminium-can shortage in Bangalore. **Data is a transparent simulation** of a real, cited event (see `docs/references.md`).

Run order: `generate_data.py` → this notebook (or `analyze_dmaic.py`).

## 0. Setup

In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
df = pd.read_csv(ROOT/'data/raw/orders.csv', parse_dates=['order_date'])
df.shape

## 1. Define / Measure — the baseline and the collapse

In [ ]:
CRISIS='2026-04-15'
ls = df[df.sugar_type=='Zero']
print('Pre-crisis OTIF (all):   %.1f%%' % (df[df.order_date< CRISIS].otif.mean()*100))
print('Low-sugar pre-crisis:    %.1f%%' % (ls[ls.order_date< CRISIS].otif.mean()*100))
print('Low-sugar CRISIS OTIF:   %.1f%%' % (ls[ls.order_date>=CRISIS].otif.mean()*100))
print('Low-sugar crisis fill:   %.1f%%' % (ls[ls.order_date>=CRISIS].delivered_cases.sum()/ls[ls.order_date>=CRISIS].ordered_cases.sum()*100))

## 2. Analyze — Pareto of crisis failure reasons (one cause dominates)

In [ ]:
crisis = df[df.order_date>=CRISIS]
p = crisis[crisis.failure_reason!='None'].failure_reason.value_counts()
(p/p.sum()*100).round(1)

## 3. Analyze — pack format is the fault line (chi-square)

In [ ]:
from scipy import stats
ct = pd.crosstab(crisis.pack_format, crisis.otif)
chi2,pval,dof,_ = stats.chi2_contingency(ct)
print('CAN crisis OTIF: %.1f%%' % (crisis[crisis.pack_format=='CAN'].otif.mean()*100))
print('PET crisis OTIF: %.1f%%' % (crisis[crisis.pack_format=='PET'].otif.mean()*100))
print('chi2=%.0f, dof=%d, p=%.3g' % (chi2,dof,pval))

## 4. Impact — lost margin

In [ ]:
print('Crisis lost margin: Rs %.2f cr' % (crisis.lost_margin_inr.sum()/1e7))
print('Crisis lost revenue: Rs %.2f cr' % (crisis.lost_revenue_inr.sum()/1e7))

## 5. Full pipeline + figures
Run the production script to regenerate all 8 figures and `results.json`:

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, str(ROOT/'src/analyze_dmaic.py')], capture_output=True, text=True).stdout)

## 6. Headline results (machine-readable)

In [ ]:
json.load(open(ROOT/'reports/results.json'))

See `reports/DMAIC_Report.md` for the full 18-section report and résumé/LinkedIn framing.